# LUAD cell fractions by molecular subtype

Load plotting settings, LUAD subtype order, colors, and the fixed server output path.

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from scipy.stats import mannwhitneyu
import matplotlib as mpl
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")


mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans"],
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.linewidth": 0.7,
        "xtick.major.width": 0.7,
        "ytick.major.width": 0.7,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "savefig.edgecolor": "white",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)
plots_dir = "/Path/to/output/cancer/LUAD/plots_luad_subtype_boxplots"
os.makedirs(plots_dir, exist_ok=True)
methods = ["SABER","DWLS", "BayesPrism"]
npg_colors = ["#3C5488", "#E64B35", "#00A087", "#4DBBD5", "#F39B7F"]
luad_order = ["TRU", "prox.-inflam", "prox.-prolif."]
luad_palette = dict(zip(luad_order, npg_colors))


## Test predicted fractions across LUAD subtypes

Match TCGA patients to SABER, DWLS, and BayesPrism predictions, compare the highest-mean subtype with the other groups using patients as statistical units, and save one PDF per method.


In [6]:
prediction_paths = {method: f"/Path/to/cancer/results/LUAD1/{method}.csv" for method in methods}
missing_prediction_files = [path for path in prediction_paths.values() if not os.path.exists(path)]
if missing_prediction_files:
    raise FileNotFoundError(f"Missing required prediction files: {missing_prediction_files}")

subtype_df = pd.read_csv("/Path/to/cancer/data/LUAD1/TCGA_LUAD_subtype.csv")
subtype_df["patient"] = subtype_df["patient"].astype(str).str[:12]
subtype_labels = subtype_df.set_index("patient")["expression_subtype"]
subtype_labels = subtype_labels[subtype_labels.isin(luad_order)]
for method in methods:
    P_pred = pd.read_csv(prediction_paths[method], index_col=0)
    P_pred.index = P_pred.index.str.replace(".", "-", regex=False).str[:12]
    common_samples = subtype_labels.index.intersection(P_pred.index).drop_duplicates()
    luad_counts = subtype_labels.loc[common_samples].value_counts().reindex(luad_order, fill_value=0)
    (fig, axes) = plt.subplots(nrows=1, ncols=4, figsize=(7, 3.4))
    axes = axes.flatten()
    for ax, ct in zip(axes, ["Malignant", "T CD8", "Macrophages", "NK cells"]):
        plot_data = pd.DataFrame(
            {
                "Proportion": P_pred.loc[common_samples, ct],
                "Subtype": subtype_labels.loc[common_samples],
            }
        ).dropna()
        plot_data["Subtype"] = pd.Categorical(
            plot_data["Subtype"], categories=luad_order, ordered=True
        )
        sns.boxplot(
            x="Subtype",
            y="Proportion",
            data=plot_data,
            order=luad_order,
            palette=luad_palette,
            ax=ax,
            width=0.58,
            linewidth=0.8,
            fliersize=0,
            saturation=0.95,
            boxprops=dict(edgecolor="black", linewidth=0.8),
            whiskerprops=dict(color="black", linewidth=0.8),
            capprops=dict(color="black", linewidth=0.8),
            medianprops=dict(color="black", linewidth=1.0),
        )
        sns.stripplot(
            x="Subtype",
            y="Proportion",
            data=plot_data,
            order=luad_order,
            ax=ax,
            color="black",
            size=2.0,
            alpha=0.35,
            jitter=0.22,
            linewidth=0,
        )
        ax.set_title(ct, fontsize=9, fontweight="normal", pad=5)
        ax.set_xlabel("")
        ax.set_ylabel("Proportion", fontsize=8)
        max_subtype = plot_data.groupby("Subtype", observed=False)["Proportion"].mean().idxmax()
        data_min = plot_data["Proportion"].min()
        data_max_value = plot_data["Proportion"].max()
        y_range = data_max_value - data_min
        if y_range == 0:
            y_range = 0.05
        h = y_range * 0.06
        line_y = data_max_value + h
        final_y = line_y
        group_max_data = plot_data[plot_data["Subtype"] == max_subtype]["Proportion"]
        for i, target_subtype in enumerate(luad_order):
            if target_subtype == max_subtype:
                continue
            group_target_data = plot_data[plot_data["Subtype"] == target_subtype]["Proportion"]
            assert len(group_max_data) > 2 and len(group_target_data) > 2
            (_, pval) = mannwhitneyu(group_max_data, group_target_data, alternative="two-sided")
            if pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = None
            if sig:
                x1 = luad_order.index(max_subtype)
                x2 = i
                ax.plot(
                    [x1, x1, x2, x2],
                    [line_y, line_y + h / 2, line_y + h / 2, line_y],
                    lw=0.8,
                    c="black",
                )
                ax.text(
                    (x1 + x2) / 2,
                    line_y + h / 2,
                    sig,
                    ha="center",
                    va="bottom",
                    fontsize=8,
                    color="black",
                )
                final_y = line_y + h / 2
                line_y += h * 1.6
        ax.set_ylim(data_min - y_range * 0.08, final_y + y_range * 0.2)
        ax.tick_params(axis="x", which="both", labelbottom=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(0.7)
        ax.spines["bottom"].set_linewidth(0.7)
        ax.tick_params(axis="both", which="major", width=0.7, length=3, pad=2)
        ax.grid(False)
    fig.suptitle(
        f"{method} - Cell Proportions by LUAD Subtype", fontsize=12, fontweight="normal", y=1.03
    )
    fig.legend(
        handles=[
            Patch(
                facecolor=luad_palette[subtype],
                edgecolor="black",
                linewidth=0.5,
                label=f"{subtype} (n={int(luad_counts.get(subtype, 0))})",
            )
            for subtype in luad_order
        ],
        loc="lower center",
        ncol=len(luad_order),
        frameon=False,
        bbox_to_anchor=(0.5, 0.01),
        handlelength=1.1,
        handleheight=0.8,
        handletextpad=0.5,
        columnspacing=1.4,
        fontsize=7,
    )
    plt.tight_layout(rect=[0, 0.1, 1, 0.95])
    plt.savefig(f"{plots_dir}/{method}_LUAD_subtype_boxplot.pdf", format="pdf", bbox_inches="tight")
    plt.close()
